In [7]:
xai_data = {

    "dataset_name": "heart",

    "model": final_model,

    "X_train": X_train,
    "X_test": X_test,

    "y_train": y_train,
    "y_test": y_test,

    "feature_names": feature_names,

    "X_full": X_scaled_df,

    "y_full": y,

    "scoring": "roc_auc",

    "train_model_fn": train_model_fn
}

NameError: name 'final_model' is not defined

In [6]:
print(type(final_model))
print(final_model.__class__.__name__)

NameError: name 'final_model' is not defined

### Standard metadata object

In [3]:
xai_data = {

    "dataset_name": DATASET_NAME,

    "model": final_model,

    "X_train": X_train,
    "X_test": X_test,

    "y_train": y_train,
    "y_test": y_test,

    "feature_names": feature_names,

    "train_model_fn": train_model_fn,

    "X_full": X_full,
    "y_full": y_full,

    "scoring": "roc_auc",

    "explanation_level": "predictor"
}

NameError: name 'final_model' is not defined

#### HbA1c adds:

In [ ]:
xai_data.update({
    "analysis_df": analysis_df,
    "X_design": X_design,
    "formula": formula
})

### Imports

In [ ]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import shap

from scipy.stats import spearmanr

from sklearn.inspection import permutation_importance

from lime.lime_tabular import LimeTabularExplainer

### Model Inspection

In [ ]:
def detect_model_family(model):

    if hasattr(model, "params"):
        return "statsmodels"

    if hasattr(model, "feature_importances_"):
        return "tree"

    if hasattr(model, "coef_"):
        return "linear"

    if "xgboost" in str(type(model)).lower():
        return "boosting"

    if "lightgbm" in str(type(model)).lower():
        return "boosting"

    return "generic"

In [ ]:
def inspect_model(model):

    info = {
        "class": model.__class__.__name__,
        "module": model.__class__.__module__,
        "predict": hasattr(model, "predict"),
        "predict_proba": hasattr(model, "predict_proba"),
        "coef": hasattr(model, "coef_"),
        "params": hasattr(model, "params"),
        "feature_importances": hasattr(
            model,
            "feature_importances_"
        )
    }

    return info

##### Unified prediction functions

In [ ]:
def get_probabilities(model, X):

    # sklearn classifiers
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    # statsmodels logistic regression
    if hasattr(model, "params"):
        return np.asarray(model.predict(X))

    raise ValueError(
        f"No probability method found for {type(model)}"
    )

In [ ]:
def get_predictions(model, X, threshold=0.5):

    probs = get_probabilities(model, X)

    return (probs >= threshold).astype(int)

### Capability Inspection

In [ ]:
def detect_capabilities(model):

    return {

        "native_coefficients":
            hasattr(model, "coef_")
            or hasattr(model, "params"),

        "native_importance":
            hasattr(model, "feature_importances_"),

        "probabilities":
            hasattr(model, "predict_proba")
            or hasattr(model, "params"),

        "permutation":
            True,

        "shap":
            True,

        "lime":
            True
    }

### XAI method selection

In [ ]:
def select_global_xai_methods(model):

    caps = detect_capabilities(model)

    methods = []

    if caps["native_coefficients"]:
        methods.append("coefficients")

    if caps["native_importance"]:
        methods.append(
            "feature_importance"
        )

    methods.append(
        "permutation_importance"
    )

    methods.append(
        "shap_importance"
    )

    return methods

### Native Importance Extraction

In [ ]:
def get_native_importance(
    model,
    feature_names=None
):

    if hasattr(model, "coef_"):

        coef = model.coef_[0]

        return pd.DataFrame({

            "Feature":
                feature_names,

            "Importance":
                np.abs(coef),

            "RawValue":
                coef,

            "Method":
                "Coefficient"

        }).sort_values(
            "Importance",
            ascending=False
        )

    if hasattr(model, "params"):

        params = model.params

        return pd.DataFrame({

            "Feature":
                params.index,

            "Importance":
                np.abs(
                    params.values
                ),

            "RawValue":
                params.values,

            "Method":
                "Coefficient"

        }).sort_values(
            "Importance",
            ascending=False
        )

    if hasattr(model, "feature_importances_"):

        return pd.DataFrame({

            "Feature":
                feature_names,

            "Importance":
                model.feature_importances_,

            "Method":
                "FeatureImportance"

        }).sort_values(
            "Importance",
            ascending=False
        )

    return None

### Universal Permutation Importance

In [ ]:
def get_permutation_importance_df(
    model,
    X_test,
    y_test,
    feature_names,
    scoring="roc_auc"
):

    perm = permutation_importance(

        model,

        X_test,

        y_test,

        n_repeats=30,

        random_state=42,

        scoring=scoring
    )

    return pd.DataFrame({

        "Feature":
            feature_names,

        "PermutationMean":
            perm.importances_mean,

        "PermutationStd":
            perm.importances_std

    }).sort_values(
        "PermutationMean",
        ascending=False
    )

### Automatic SHAP Explainer

In [ ]:
def build_shap_explainer(model, X_background):

    family = detect_model_family(model)

    try:

        if family == "tree":
            return shap.TreeExplainer(model)

        if family == "linear":
            return shap.LinearExplainer(
                model,
                X_background
            )

        return shap.Explainer(
            lambda x: get_probabilities(model, x),
            X_background
        )

    except Exception:

        return shap.Explainer(
            lambda x: get_probabilities(model, x),
            X_background
        )

### SHAP Importance

In [ ]:
def get_shap_importance(
    model,
    X_train,
    X_test,
    feature_names
):

    explainer = build_shap_explainer(
        model,
        X_train
    )

    shap_values = explainer(X_test)

    values = shap_values.values

    if values.ndim == 3:

        values = values[:, :, 1]

    shap_df = pd.DataFrame({

        "Feature":
            feature_names,

        "MeanAbsSHAP":
            np.abs(values).mean(axis=0)

    }).sort_values(
        "MeanAbsSHAP",
        ascending=False
    )

    return (
        explainer,
        shap_values,
        shap_df
    )

### Rank Comparison

In [ ]:
def compare_importance_ranks(
    native_df,
    shap_df,
    perm_df
):

    shap_rank = shap_df.copy()

    shap_rank["SHAPRank"] = np.arange(
        1,
        len(shap_rank) + 1
    )

    perm_rank = perm_df.copy()

    perm_rank["PermutationRank"] = np.arange(
        1,
        len(perm_rank) + 1
    )

    if native_df is None:

        return (
            shap_rank
            .merge(
                perm_rank,
                on="Feature"
            )
        )

    native_rank = native_df.copy()

    native_rank["NativeRank"] = np.arange(
        1,
        len(native_rank) + 1
    )

    return (
        native_rank
        .merge(
            shap_rank,
            on="Feature"
        )
        .merge(
            perm_rank,
            on="Feature"
        )
    )

### Representative Cases

In [ ]:
def get_representative_cases(
    model,
    X_test,
    y_test,
    threshold=0.5
):

    probs = get_probabilities(
        model,
        X_test
    )

    preds = (
        probs >= threshold
    ).astype(int)

    case_df = X_test.copy()

    case_df["y_true"] = np.asarray(y_test)
    case_df["y_pred"] = preds
    case_df["y_prob"] = probs

    case_df["dist"] = np.abs(
        probs - threshold
    )

    tp = case_df[
        (case_df.y_true == 1)
        &
        (case_df.y_pred == 1)
    ].nlargest(1, "y_prob")

    tn = case_df[
        (case_df.y_true == 0)
        &
        (case_df.y_pred == 0)
    ].nsmallest(1, "y_prob")

    fp = case_df[
        (case_df.y_true == 0)
        &
        (case_df.y_pred == 1)
    ].nlargest(1, "y_prob")

    fn = case_df[
        (case_df.y_true == 1)
        &
        (case_df.y_pred == 0)
    ].nsmallest(1, "y_prob")

    borderline = case_df.nsmallest(
        1,
        "dist"
    )

    tp["CaseType"] = "TP"
    tn["CaseType"] = "TN"
    fp["CaseType"] = "FP"
    fn["CaseType"] = "FN"
    borderline["CaseType"] = "Borderline"

    return pd.concat([
        tp,
        tn,
        fp,
        fn,
        borderline
    ]).drop_duplicates()

### Local SHAP

In [ ]:
def explain_cases_shap(
    explainer,
    X_test,
    rep_cases
):

    for idx in rep_cases.index:

        shap_case = explainer(
            X_test.loc[[idx]]
        )

        shap.plots.waterfall(
            shap_case[0],
            show=False
        )

        plt.show()

### Local LIME

In [ ]:
def explain_cases_lime(
    model,
    X_train,
    X_test,
    rep_cases,
    feature_names
):

    lime_explainer = LimeTabularExplainer(
        training_data=np.asarray(X_train),
        feature_names=feature_names,
        mode="classification"
    )

    results = []

    for idx in rep_cases.index:

        exp = lime_explainer.explain_instance(
            np.asarray(X_test.loc[idx]),
            lambda x: np.column_stack([
                1 - get_probabilities(model, x),
                get_probabilities(model, x)
            ])
        )

        for rule, weight in exp.as_list():

            results.append({
                "CaseIndex": idx,
                "Rule": rule,
                "Weight": weight
            })

    return pd.DataFrame(results)

### Stability Analysis

In [ ]:
def run_stability_analysis(
    train_model_fn,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_names,
    seeds=[1,2,3,4,5]
):

    rows = []

    for seed in seeds:

    model_s, X_test_s, y_test_s = train_model_fn(seed)

    perm = permutation_importance(
        model_s,
        X_test_s,
        y_test_s,
        n_repeats=10,
        random_state=seed
    )

        temp = pd.DataFrame({
            "Feature": feature_names,
            "Importance":
                perm.importances_mean,
            "Seed": seed
        })

        rows.append(temp)

    stability_df = pd.concat(rows)

    return stability_df

### Export Function

In [ ]:
def export_xai_results(
    results,
    out_dir
):

    os.makedirs(
        out_dir,
        exist_ok=True
    )

    for name, obj in results.items():

        if isinstance(
            obj,
            pd.DataFrame
        ):

            obj.to_csv(
                os.path.join(
                    out_dir,
                    f"{name}.csv"
                ),
                index=False
            )

# Global XAI runner

In [ ]:
def run_global_xai(
    model,
    X_train,
    X_test,
    y_test,
    feature_names,
    scoring=xai_data["scoring"]
):

    native_df = get_native_importance(
        model,
        feature_names
    )

    perm_df = get_permutation_importance_df(
        model,
        X_test,
        y_test,
        feature_names,
        scoring
    )

    (
        explainer,
        shap_values,
        shap_df

    ) = get_shap_importance(

        model,

        X_train,

        X_test,

        feature_names
    )

    rank_compare = compare_importance_ranks(

        native_df,

        shap_df,

        perm_df
    )

    return {

        "native_df":
            native_df,

        "perm_df":
            perm_df,

        "shap_df":
            shap_df,

        "rank_compare":
            rank_compare,

        "explainer":
            explainer,

        "shap_values":
            shap_values
    }

# Main notebook call

In [ ]:
model = xai_data["model"]

model_info = inspect_model(model)

print(model_info)

global_results = run_global_xai(

    model,

    xai_data["X_train"],

    xai_data["X_test"],

    xai_data["y_test"],

    xai_data["feature_names"]
)

rep_cases = get_representative_cases(

    model,

    xai_data["X_test"],

    xai_data["y_test"]
)

explain_cases_shap(

    global_results["explainer"],

    xai_data["X_test"],

    rep_cases
)

lime_df = explain_cases_lime(

    model,

    xai_data["X_train"],

    xai_data["X_test"],

    rep_cases,

    xai_data["feature_names"]
)

stability_df = run_stability_analysis(

    xai_data["train_model_fn"],

    xai_data["X_train"],
    xai_data["X_test"],

    xai_data["y_train"],
    xai_data["y_test"],

    xai_data["feature_names"]
)

In [ ]:
xai_results = {

    "dataset_name":
        xai_data["dataset_name"],

    "model_family":
        detect_model_family(model),

    "model_info":
        model_info,

    "native_importance":
        global_results["native_df"],

    "permutation_importance":
        global_results["perm_df"],

    "shap_importance":
        global_results["shap_df"],

    "rank_compare":
        global_results["rank_compare"],

    "representative_cases":
        rep_cases,

    "lime_cases":
        lime_df,

    "stability":
        stability_df
}

export_xai_results(
    xai_results,
    out_dir
)